# CryptoWatch - Apache Spark Analysis
**Kelompok 7**
- Ananda Widi Alrafi
- Nafis Faqih A.M.
- Tiara Fatimah
- Zahra Hafizhah
- Mohamad Arkan Zahir A.

In [ ]:
import os
import json
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, count, max as spark_max, min as spark_min,
    stddev, abs as spark_abs, hour, to_timestamp, round as spark_round, cast
)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("CryptoWatch Analysis") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:8020") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark Session Berhasil Dibuat!")

## 1. Analisis Statistik Harga Per Koin
Menghitung rata-rata, harga tertinggi, terendah, dan volatilitas (standar deviasi).

In [ ]:
HDFS_API_PATH = "hdfs://localhost:8020/data/crypto/api/"

try:
    df_api = spark.read.option("multiLine", True).json(HDFS_API_PATH)
    print(f"Total records API: {df_api.count()}")
    
    df_api.createOrReplaceTempView("crypto_prices")
    
    stats = df_api.groupBy("symbol").agg(
        spark_round(avg("price_usd"), 2).alias("avg_price"),
        spark_round(spark_max("price_usd"), 2).alias("max_price"),
        spark_round(spark_min("price_usd"), 2).alias("min_price"),
        spark_round(stddev("price_usd"), 2).alias("stddev_price"),
        count("*").alias("total_records")
    ).orderBy(col("avg_price").desc())
    
    stats.show()
except Exception as e:
    print("Belum ada data di HDFS atau error:", e)

## 2. Volatilitas Per Jam (Menggunakan Spark SQL)
Menganalisis pergerakan absolut harga per jam untuk menentukan jam trading paling aktif.

In [ ]:
try:
    volatility = spark.sql("""
        SELECT 
            HOUR(TO_TIMESTAMP(timestamp)) as jam,
            ROUND(AVG(ABS(change_24h)), 4) as avg_volatility,
            COUNT(*) as jumlah_data,
            ROUND(MAX(ABS(change_24h)), 4) as max_volatility
        FROM crypto_prices
        GROUP BY HOUR(TO_TIMESTAMP(timestamp))
        ORDER BY jam
    """)
    
    volatility.show()
except Exception as e:
    print("Error:", e)

## 3. Volume Berita Per Jam
Menganalisis jumlah artikel berita yang rilis setiap jamnya dari sumber RSS.

In [ ]:
HDFS_RSS_PATH = "hdfs://localhost:8020/data/crypto/rss/"

try:
    df_rss = spark.read.option("multiLine", True).json(HDFS_RSS_PATH)
    print(f"Total records RSS: {df_rss.count()}")
    
    df_rss.createOrReplaceTempView("crypto_news")
    
    news_volume = spark.sql("""
        SELECT 
            HOUR(TO_TIMESTAMP(timestamp)) as jam,
            COUNT(*) as jumlah_artikel,
            COUNT(DISTINCT source) as jumlah_sumber
        FROM crypto_news
        GROUP BY HOUR(TO_TIMESTAMP(timestamp))
        ORDER BY jumlah_artikel DESC
    """)
    
    news_volume.show()
except Exception as e:
    print("Error:", e)

## 4. (BONUS) K-Means Clustering Harga Kripto menggunakan Spark MLlib
Menggunakan KMeans untuk mengelompokkan koin secara otomatis berdasarkan kemiripan harga dan pergerakan 24 jam.

In [ ]:
try:
    ml_data = df_api.select(
        col("symbol"), 
        col("price_usd").cast("float").alias("price"), 
        col("change_24h").cast("float").alias("change")
    ).na.drop()

    if ml_data.count() > 3:
        assembler = VectorAssembler(inputCols=["price", "change"], outputCol="features")
        dataset = assembler.transform(ml_data)

        kmeans = KMeans().setK(3).setSeed(1).setFeaturesCol("features")
        model = kmeans.fit(dataset)
        predictions = model.transform(dataset)
        
        print("Pusat Cluster (Centroids):")
        centers = model.clusterCenters()
        for i, center in enumerate(centers):
            print(f"  Cluster {i}: Harga=${center[0]:.2f}, Perubahan={center[1]:.2f}%")
            
        cluster_stats = predictions.groupBy("prediction").agg(
            count("*").alias("jumlah_titik"),
            spark_round(avg("price"), 2).alias("avg_price")
        ).orderBy("prediction")
        
        cluster_stats.show()
    else:
        print("Data terlalu sedikit untuk clustering")
except Exception as e:
    print("Error MLlib:", e)